# ARCH 6133 Places / Platforms
## NYC Index Builder: Indexes Are Arguments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danmillr/places-platforms/blob/main/tutorials/ARCH6133_NYC_Index_Builder.ipynb)

---

### What this notebook does

This notebook walks you through building urban indexes for New York City. By the end, you will have:
- Pulled tract-level Census ACS data and four NYC Open Data sources into a single workspace
- Spatially joined point data (trees, wi-fi, subway, 311) to two competing geographies (Census tract and NTA)
- Derived an emergent boundary set from 311 clusters and compared it to administrative boundaries
- Worked through four pre-built example indexes (Green Equity, Transit and Digital Equity, Housing Stress, Neighborhood Cohesion Proxy)
- Built your own index with interactive widgets, defended every choice in an auto-generated Methodology Card, and exported it as GeoJSON and CSV
- Sat with several rounds of reflection prompts that ask who is helped and who is harmed by each choice

This notebook lives in the `tutorials/` folder of the [places-platforms](https://github.com/danmillr/places-platforms) repository. It follows the same naming pattern as `ARCH6133_POI_Data.ipynb` and `ARCH6133_Street_View_Lab.ipynb`: an `ARCH6133_` prefix, a `Title_Case` description, the `.ipynb` extension, and an **Open in Colab** badge in the first cell that points to `main`.

### How to get a free Census API key

1. Go to https://api.census.gov/data/key_signup.html
2. Fill in your name, organization (Cornell University is fine), and email
3. The key arrives by email in under a minute
4. In Colab, open the **Secrets** panel (key icon in the left sidebar)
5. Add a new secret named `CENSUS_API_KEY`, paste the key as the value, and toggle **Notebook access** ON

### Data sources and update frequencies

| Source | Update Frequency | Known Limitations |
|---|---|---|
| ACS 5-year (Census API) | Annually, lagged ~1 year | High margin of error at block-group scale |
| PLUTO (NYC Open Data) | ~2x per year | Lags zoning changes; some owner fields stale |
| Street Tree Census (NYC OD) | 2015 snapshot (next survey 2025) | Streets only — no parks, no private trees |
| Wi-Fi Hotspots (NYC OD) | Quarterly | Public hotspots only; overcounts in tourist zones |
| Heat Vulnerability Index (DOHMH) | Annual | Pre-aggregated at NTA; methodology fixed |
| 311 Service Requests (NYC OD) | Daily | Reflects **who calls**, not where problems exist |
| Subway Stations / Entrances (MTA via **NY State** Open Data) | Periodic | ADA flag can lag the actual elevator status |
| NTA Boundaries (DCP / NYC OD) | Major release per decennial Census | 2020 vintage used here |

### How to enable ipywidgets in Colab

Module 0 runs `output.enable_custom_widget_manager()` for you. If a slider or dropdown does not render later, restart the runtime and re-run Module 0 from the top.

### Estimated runtime

Module 0: < 1 minute
Module 1: 1-2 minutes (tract + NTA boundary download)
Module 2: 5-10 minutes (multiple Open Data pulls, sampled)
Module 3: 1-2 minutes
Module 4: 1-2 minutes
Module 5: 2-3 minutes (four indexes)
Module 6: depends on you
Module 7: < 1 minute

Total: ~15-25 minutes for an end-to-end run.

### The argument

This notebook takes a position: **indexes are arguments, not measurements**. The point of this exercise is not to produce a correct index but to produce a legible one — where every choice is visible, every assumption is named, and every weight is something you could defend in a community meeting. Read the framing cell in Module 1 before any of the code, and return to it whenever you are tempted to treat a number as the truth about a place.


---

## Module 0 — Setup and Configuration

Run every cell in this module in order. If your runtime resets later, come back here and re-run from the top. The Drive mount, package installs, API key handle, and helper functions defined here are used by every subsequent module.


### Mount Google Drive

All outputs of this notebook — raw data, aggregated GeoJSON, choropleth maps, methodology cards — land in a project folder in your Drive. Run the cell and accept the permission dialog.

In [ ]:
# Mount your personal Google Drive into the Colab filesystem.
from google.colab import drive
drive.mount('/content/drive')

### Install required packages

One consolidated install. Some of these wheels are big — geopandas in particular pulls in proj/gdal — so this cell typically takes 60-90 seconds in Colab.

In [ ]:
# One consolidated install. Quiet flag cuts dependency-resolution noise.
!pip install -q requests pandas geopandas folium mapclassify matplotlib seaborn scikit-learn ipywidgets tqdm census shapely

### Store your Census API key in Colab Secrets

The NYC Open Data endpoints used in this notebook do **not** require authentication, but the US Census API does.

To get a key:
1. Visit https://api.census.gov/data/key_signup.html (free, takes a minute)
2. Wait for the email confirmation
3. In Colab, click the key icon in the left sidebar (Secrets)
4. Click 'Add new secret'
5. Set Name to: `CENSUS_API_KEY`
6. Paste the key as the Value
7. Toggle 'Notebook access' to ON
8. Return here and run the cell below

Your key never appears in any code or saved file.

In [ ]:
# Pull the Census API key from Colab Secrets and validate.
from google.colab import userdata

CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')

if not CENSUS_API_KEY:
    print("ERROR: CENSUS_API_KEY not found in Colab Secrets.")
    print("Follow the instructions above to add it, then re-run this cell.")
    raise ValueError("Missing CENSUS_API_KEY")
else:
    print("Census API key loaded. NYC Open Data endpoints do not require a key.")

### Configure outputs

Edit the two values below if you want different defaults. `TOP_N` controls how many top-scoring (and bottom-scoring) areas appear in every ranking output.

In [ ]:
# Project configuration. Edit and re-run if you change defaults.
OUTPUT_FOLDER = '/content/drive/MyDrive/NYCIndexLab/'
TOP_N = 10

import os

SUBFOLDERS = ['data', 'maps', 'exports', 'cards']
for sub in [''] + SUBFOLDERS:
    os.makedirs(os.path.join(OUTPUT_FOLDER, sub), exist_ok=True)

# Enable ipywidgets in Colab so Module 4 and Module 6 render correctly.
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

print(f"Setup complete. Output folder ready at {OUTPUT_FOLDER}.")
for sub in SUBFOLDERS:
    print(f"  - {sub}/")

### Reusable fetch helpers

Two helpers we will reuse everywhere:
- `fetch_nyc_open_data(endpoint, params, max_rows)` — wraps Socrata's `data.cityofnewyork.us/resource/<id>.json` endpoints with pagination and plain-English error messages.
- `fetch_census(variables, geography, year)` — pulls ACS 5-year estimates for the five NYC counties via the `census` Python library, then joins them to TIGER tract geometry.

Both return a pandas DataFrame (the Census helper returns a GeoDataFrame with a `geometry` column).

In [ ]:
# Reusable fetch helpers.
import requests
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

# Five NYC county FIPS codes for the Census API.
NYC_COUNTIES = {
    "005": "Bronx",
    "047": "Kings",       # Brooklyn
    "061": "New York",    # Manhattan
    "081": "Queens",
    "085": "Richmond",    # Staten Island
}

def fetch_nyc_open_data(endpoint, params=None, max_rows=50000, page_size=5000,
                        domain="data.cityofnewyork.us"):
    """Page through a Socrata resource and return a DataFrame of all rows.

    Defaults to NYC Open Data. Pass domain='data.ny.gov' for NY State datasets
    (e.g. MTA subway entrances, which were retired from NYC Open Data)."""
    base_url = f"https://{domain}/resource/{endpoint}"
    params = dict(params or {})
    collected = []
    progress = tqdm(total=max_rows, desc=f"Fetching {endpoint}", unit="rows")
    for offset in range(0, max_rows, page_size):
        batch_limit = min(page_size, max_rows - offset)
        params["$limit"] = batch_limit
        params["$offset"] = offset
        try:
            response = requests.get(base_url, params=params, timeout=60)
            response.raise_for_status()
            batch = response.json()
        except Exception as fetch_error:
            print(f"  Fetch stopped at offset {offset}: {fetch_error}")
            break
        if not isinstance(batch, list) or len(batch) == 0:
            break
        collected.extend(batch)
        progress.update(len(batch))
        if len(batch) < batch_limit:
            break
    progress.close()
    return pd.DataFrame(collected)

# Cache the NY State tract GeoDataFrame so we only download it once per session.
_TIGER_TRACTS_CACHE = {}

def _load_nyc_tract_geometry(year=2022):
    """Download and cache TIGER cartographic-boundary tracts for NYC."""
    cache_key = ("tracts", year)
    if cache_key in _TIGER_TRACTS_CACHE:
        return _TIGER_TRACTS_CACHE[cache_key]
    url = f"https://www2.census.gov/geo/tiger/GENZ{year}/shp/cb_{year}_36_tract_500k.zip"
    print(f"Downloading TIGER tract geometry for NY State ({year})...")
    tracts = gpd.read_file(url)
    tracts = tracts[tracts["COUNTYFP"].isin(NYC_COUNTIES.keys())].copy()
    tracts["GEOID"] = tracts["GEOID"].astype(str)
    _TIGER_TRACTS_CACHE[cache_key] = tracts
    return tracts

def fetch_census(variables, geography="tract", year=2022):
    """Pull ACS 5-year estimates for NYC counties and attach geometry."""
    from census import Census
    census_client = Census(CENSUS_API_KEY, year=year)
    all_rows = []
    for county_fips in NYC_COUNTIES:
        try:
            if geography == "tract":
                county_rows = census_client.acs5.state_county_tract(
                    variables, "36", county_fips, Census.ALL)
            else:
                county_rows = census_client.acs5.state_county(
                    variables, "36", county_fips)
            all_rows.extend(county_rows)
        except Exception as census_error:
            print(f"Census fetch failed for county {county_fips}: {census_error}")
    df = pd.DataFrame(all_rows)
    if df.empty:
        print("Census API returned no rows. Check your API key and variable codes.")
        return df
    if geography == "tract":
        df["GEOID"] = df["state"].astype(str) + df["county"].astype(str) + df["tract"].astype(str)
        tracts = _load_nyc_tract_geometry(year=year)
        merged = tracts.merge(df, on="GEOID", how="left")
        return merged
    return df

print("Helpers fetch_nyc_open_data() and fetch_census() ready.")

---

## Module 1 — Understanding the Data Landscape

This module does not produce a map. It produces **understanding**. Read the four sections below before you fetch a single row of data. The decisions you make about which variables to use and at what geography are more consequential than any line of code you will write afterward.

### Quantification, briefly and critically

Reducing urban conditions to numbers always involves loss. A median household income for a Census tract collapses thousands of lives — full-time workers, retirees, students, undocumented workers paid in cash, people between jobs, multi-generational households — into a single number. That number can be useful. It can also be wrong about every household in the tract simultaneously. The texture, the history, the relationships of power between neighbors and landlords, the friction of being in a place rather than next to it: none of that aggregates cleanly into a column.

The history of index-building in urban policy is long, and it cuts both ways. HUD's **Affirmatively Furthering Fair Housing** index was designed to surface racial and economic segregation so federal funding could be targeted at correcting it. The CDC's **Social Vulnerability Index** is used to direct disaster response toward communities least able to recover. The **Heat Vulnerability Index** here in NYC was built by the Department of Health to identify neighborhoods at highest risk of heat-related mortality. These are useful tools. They also have a darker shadow: redlining maps were a kind of index, scoring neighborhoods on perceived "stability" and using those scores to justify decades of disinvestment. An index that allocates resources is also an index that denies them. Indexes built to help can be repurposed to harm. Indexes built to harm sometimes get repackaged as help.

The geography you choose to aggregate into is itself a political choice. Census tracts were drawn to contain roughly equal populations within a county, not to reflect how neighborhoods are actually experienced. Their boundaries can run down the middle of a block, splitting communities that share a school, a corner store, and a bus stop. Neighborhood Tabulation Areas were drawn by NYC's Department of City Planning to *approximate* recognized neighborhood names, which means they encode the planner's idea of where Bed-Stuy ends and Crown Heights begins — an idea that not every resident shares. The act of putting a line around a place is always partial, and the line is always doing work for someone.

**The most dangerous index is one that looks objective. Your job in this notebook is to make your assumptions visible.**

### Census data, in plain terms

The American Community Survey (ACS) is a continuous sample survey conducted by the Census Bureau. Unlike the **decennial Census** — which attempts to count every person once every ten years and asks a small set of questions — the ACS samples about 3.5 million addresses each year and asks dozens of questions about income, education, commute, language, disability, and housing.

Because it is a sample, every ACS estimate comes with a **margin of error (MOE)**. The smaller the geography you ask about, the smaller the sample, the bigger the MOE. At the **block group** scale (about 600-3000 people), MOEs can swallow the estimate whole — a "median income of \$45,000 ± \$28,000" tells you very little. At the **tract** scale (1,200-8,000 people) MOEs are usually tolerable but still worth checking.

You will see two flavors of ACS estimates: **1-year** (only available for areas with 65,000+ residents, more current, higher variance) and **5-year** (available everywhere, smoother, but reflects an average across five years rather than a single year). This notebook uses **5-year 2022** estimates, which span 2018-2022.

Browse the variable list at https://api.census.gov/data/2022/acs/acs5/variables.html — there are over 27,000 variables. For every estimate variable (suffix `E`) there is a corresponding margin-of-error variable (suffix `M`). **We will fetch both and flag any tract where MOE / estimate > 30%** as `high_uncertainty`.

### NYC Open Data, in plain terms

NYC Open Data is a portal at https://data.cityofnewyork.us where city agencies publish datasets under **Local Law 11 of 2012**. Most datasets are queryable via the Socrata API — no key required for the volumes we will use.

Datasets relevant to this notebook:

| Dataset | What it captures | Critical caveat |
|---|---|---|
| **PLUTO** | One row per tax lot — zoning, building class, floor area, basement code, lat/lng | "Basement" is a numeric code (0-5), not a boolean — interpret carefully |
| **Street Tree Census 2015** | Every street tree the city could find, with species, diameter, condition | Streets only — parks, private yards, and post-2015 plantings are absent |
| **NYC Wi-Fi Hotspots** | Public broadband access points | Tourist-heavy areas (Times Sq, Bryant Park) are overrepresented |
| **Heat Vulnerability Index** | Pre-computed DOHMH index, per NTA | Use as a comparison layer, not a building block |
| **311 Service Requests** | Every 311 call, with type, location, date | A map of **who calls**, not where conditions are worst |
| **Subway Station Entrances** | Entrance points, ADA compliance | ADA fields can lag the actual built condition |
| **FEMA Flood Zones** | 100-year floodplain polygons | Static maps in an era of changing risk |

Joining across these datasets often means crossing geographies. A 311 call is a point. PLUTO is per lot. ACS is per tract. NTAs aggregate many tracts. **Every join introduces aggregation error.** A point falling exactly on a tract boundary is assigned to one tract or the other on the basis of arbitrary tie-breaking; an NTA-level statistic averages over patterns that may be visible at tract scale and hides them.

For the purposes of this notebook we will deliberately compute the same aggregations at **two** geographies — tract and NTA — so you can see how the choice of geography changes the story.

### Geographic units, compared

| Unit | Typical population | Drawn by | Best for | Worst for |
|---|---|---|---|---|
| **Block group** | 600 – 3,000 | Census Bureau | Hyperlocal analysis | High-MOE problems make most ACS variables unusable |
| **Census tract** | 1,200 – 8,000 | Census Bureau | The workhorse — most urban analysis lives here | Boundaries are old; can split lived neighborhoods |
| **NTA (Neighborhood Tabulation Area)** | 15,000 – 100,000 | NYC DCP | Communicating findings with neighborhood names people recognize | Masks internal variation |
| **Community District** | ~150,000 | NYC Charter | Aligning with CB governance and agency budgeting | Very coarse — within-CD inequality invisible |
| **Borough** | 500,000 – 2.5M | History | High-level rhetorical comparisons | Almost any granular question dissolves at this scale |

Now we fetch the two geographies we will use most — Census tracts and 2020 NTAs — and display them side by side so you can see what each one looks like over the city you walk.

In [ ]:
# Module 1 — fetch tract and NTA boundaries, display side by side.
import os
import geopandas as gpd
import matplotlib.pyplot as plt

# Pull NYC tracts (loaded by the helper, then cached for the session).
nyc_tracts = _load_nyc_tract_geometry(year=2022)
print(f"Loaded {len(nyc_tracts)} Census tracts for NYC")

# Pull 2020 NTAs from NYC Open Data. The Socrata GeoJSON endpoint returns
# polygons directly, so we read it with geopandas.
NTA_GEOJSON_URL = "https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson?$limit=500"
nyc_ntas = gpd.read_file(NTA_GEOJSON_URL)
print(f"Loaded {len(nyc_ntas)} 2020 NTAs from NYC Open Data")

# Make sure both layers are in the same CRS for display.
nyc_tracts = nyc_tracts.to_crs(epsg=4326)
nyc_ntas = nyc_ntas.to_crs(epsg=4326)

# Side-by-side figure.
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
nyc_tracts.plot(ax=axes[0], facecolor="none", edgecolor="#333333", linewidth=0.3)
axes[0].set_title(f"Census tracts ({len(nyc_tracts)})")
axes[0].set_axis_off()
nyc_ntas.plot(ax=axes[1], facecolor="none", edgecolor="#3C4ED6", linewidth=0.5)
axes[1].set_title(f"Neighborhood Tabulation Areas ({len(nyc_ntas)})")
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

# Save both as GeoJSON for downstream modules.
tracts_path = os.path.join(OUTPUT_FOLDER, "data", "nyc_tracts.geojson")
ntas_path   = os.path.join(OUTPUT_FOLDER, "data", "nyc_ntas.geojson")
nyc_tracts.to_file(tracts_path, driver="GeoJSON")
nyc_ntas.to_file(ntas_path, driver="GeoJSON")
print(f"\nSaved tract boundaries: {tracts_path}")
print(f"Saved NTA boundaries:   {ntas_path}")

**Look at the two maps. Find your neighborhood.** Which boundary feels more real to you? Which one matches the route you walk to the bodega, the corner you wait for the bus on, the block where you say "I live around here"?

That feeling — the sense that one boundary maps onto your daily life and the other does not — is itself data. It tells you something about which geography will tell honest stories and which will obscure them. Hold that intuition. We will come back to it.

### Emergent geographies, briefly and critically

Administrative boundaries — tracts, NTAs, community districts — are not the only way to carve up a city. **Clustering algorithms** can derive boundaries from the data itself, grouping locations that are similar in measurable ways instead of locations that share a politically determined edge.

In Module 4 we will run K-Means on 311 complaint locations and draw the convex hull of each cluster. The result is a kind of map of where particular kinds of urban frustration concentrate. It will look very different from the NTA map. Some of those differences will be revealing.

Some will be misleading. The NYT's [Neighborhood Boundaries project](https://www.nytimes.com/interactive/2023/upshot/analog-neighborhoods.html) approached this same question by *asking people directly* — surveying residents about where they thought their neighborhood ended. That kind of bottom-up data is not available to us through public APIs. What we have instead is **311 calls**, which reflect who picks up the phone (or opens the app), which correlates strongly with income, English fluency, digital literacy, and trust in city agencies. Higher-income, more connected neighborhoods generate more 311 calls per capita. A boundary drawn from 311 clusters is partly a boundary drawn from civic engagement, which is partly a boundary drawn from race and income.

We will draw the boundaries anyway. But we will be honest about what they are.

---

## Module 2 — Fetch and Explore the Data

Now we pull every dataset the rest of the notebook will use. For each one you will see a short markdown intro (what it measures, who publishes it, what to watch out for), then a fetch cell, then a small summary of what came back.

Several of these queries are paginated and sampled — fetching every PLUTO lot or every 311 call from the last twelve months would be many gigabytes and minutes. The defaults are tuned for a teaching workflow; raise `max_rows` if you need a fuller pull for your final project.

**Module 2 depends on:** Module 0 (helpers, key) and Module 1 (boundaries are not strictly required but it is useful to have them on disk).

### ACS 5-year (2022) at tract level

These variables anchor every income / housing / commute calculation in the rest of the notebook. We fetch the estimate (`E` suffix) and the margin of error (`M` suffix) together, then flag any tract where MOE exceeds 30% of the estimate as `high_uncertainty`.

Variables:
- `B19013_001E` — Median household income
- `B25071_001E` — Median gross rent as percentage of household income
- `B08301_001E` / `B08301_010E` — Total commuters / public transit commuters
- `B01001_001E` — Total population
- `B17001_002E` — Population below poverty line
- `B25002_003E` — Vacant housing units
- `B15003_022E` — Adults with a bachelor's degree
- `C17002_001E` / `C17002_002E` / `C17002_003E` — Income-to-poverty-ratio bins


In [ ]:
# Fetch ACS variables with corresponding margin-of-error columns.
ACS_ESTIMATE_VARIABLES = [
    "B19013_001E", "B25071_001E",
    "B08301_001E", "B08301_010E",
    "B01001_001E", "B17001_002E", "B25002_003E", "B15003_022E",
    "C17002_001E", "C17002_002E", "C17002_003E",
]
# Margin-of-error variables are the same code with E replaced by M.
ACS_MOE_VARIABLES = [v[:-1] + "M" for v in ACS_ESTIMATE_VARIABLES]

acs_gdf = fetch_census(ACS_ESTIMATE_VARIABLES + ACS_MOE_VARIABLES,
                       geography="tract", year=2022)

# Cast estimate / MOE columns to numeric.
for col in ACS_ESTIMATE_VARIABLES + ACS_MOE_VARIABLES:
    if col in acs_gdf.columns:
        acs_gdf[col] = pd.to_numeric(acs_gdf[col], errors="coerce")

# Flag tracts where the MOE exceeds 30% of the estimate for income or rent.
def _flag_uncertainty(row):
    """Mark the row as high_uncertainty if any focal MOE ratio is > 0.3."""
    focal_pairs = [("B19013_001E", "B19013_001M"),
                   ("B25071_001E", "B25071_001M")]
    for estimate_col, moe_col in focal_pairs:
        estimate = row.get(estimate_col)
        moe = row.get(moe_col)
        if pd.notna(estimate) and pd.notna(moe) and estimate > 0:
            if moe / estimate > 0.3:
                return True
    return False

acs_gdf["high_uncertainty"] = acs_gdf.apply(_flag_uncertainty, axis=1)

print(f"ACS rows: {len(acs_gdf)}")
print(f"Columns: {list(acs_gdf.columns)[:20]}...")
print(f"High-uncertainty tracts: {acs_gdf['high_uncertainty'].sum()} "
      f"({100*acs_gdf['high_uncertainty'].mean():.1f}%)")
print()
print("First 3 rows of key columns:")
display(acs_gdf[["GEOID", "B19013_001E", "B25071_001E",
                 "B01001_001E", "high_uncertainty"]].head(3))

# Save to disk (drop geometry for the CSV).
acs_csv = os.path.join(OUTPUT_FOLDER, "data", "acs_tracts.csv")
acs_gdf.drop(columns="geometry").to_csv(acs_csv, index=False)
acs_gdf.to_file(os.path.join(OUTPUT_FOLDER, "data", "acs_tracts.geojson"),
                driver="GeoJSON")
print(f"\nSaved: {acs_csv}")

### PLUTO

PLUTO is the city's per-lot land-use database, maintained by the Department of City Planning. One row per tax lot, ~860,000 rows total. We use it primarily for the **basement** field (a numeric code that lets us flag lots with below-grade space, as a rough flood-vulnerability proxy) and lot area for tree-density normalization.

Default fetch is capped at 200,000 lots, sampled across NYC. Raise the cap if you need full coverage.

In [ ]:
# PLUTO — capped sample.
# Note: the basement code lives in column `bsmtcode` (the older `basement`
# alias was removed). Same semantic: 0 none, 1 below-grade unfinished,
# 2 below-grade finished, 3 above-grade unfinished, 4 above-grade finished,
# 5 unknown.
PLUTO_COLUMNS = ["bbl", "borough", "block", "lot", "address",
                 "zonedist1", "bldgclass", "numfloors", "numbldgs",
                 "unitstotal", "lotarea", "bldgarea", "comarea", "resarea",
                 "officearea", "retailarea", "bsmtcode",
                 "latitude", "longitude"]

pluto_df = fetch_nyc_open_data(
    "64uk-42ks.json",
    params={"$select": ",".join(PLUTO_COLUMNS),
            "$where": "latitude IS NOT NULL AND longitude IS NOT NULL"},
    max_rows=200000,
)

# Cast numeric columns.
for numeric_col in ["lotarea", "bldgarea", "numfloors", "unitstotal",
                    "latitude", "longitude"]:
    if numeric_col in pluto_df.columns:
        pluto_df[numeric_col] = pd.to_numeric(pluto_df[numeric_col],
                                              errors="coerce")

pluto_df["below_grade"] = pluto_df["bsmtcode"].astype(str).isin(["1", "2"])

print(f"PLUTO rows: {len(pluto_df)}")
print(f"Below-grade lots: {pluto_df['below_grade'].sum()} "
      f"({100*pluto_df['below_grade'].mean():.1f}%)")
print()
print("First 3 rows:")
display(pluto_df.head(3))
print()
print("Null counts (focal columns):")
print(pluto_df[["latitude", "longitude", "basement", "lotarea"]].isna().sum())

pluto_csv = os.path.join(OUTPUT_FOLDER, "data", "pluto_sample.csv")
pluto_df.to_csv(pluto_csv, index=False)
print(f"\nSaved: {pluto_csv}")

### Street Tree Census 2015

Run by the Parks Department with thousands of volunteers — every street tree in NYC was logged, identified, measured (`tree_dbh` is diameter at breast height in inches), and assessed for condition. The next survey is 2025-2026. **This dataset captures street trees only** — parks, private gardens, and trees planted after 2015 are absent.

In [ ]:
# Street Tree Census 2015.
TREE_COLUMNS = ["tree_id", "spc_common", "tree_dbh", "status", "health",
                "address", "latitude", "longitude",
                "boroname", "nta", "nta_name"]
trees_df = fetch_nyc_open_data(
    "uvpi-gqnh.json",
    params={"$select": ",".join(TREE_COLUMNS),
            "$where": "status = 'Alive'"},
    max_rows=100000,
)
trees_df["tree_dbh"] = pd.to_numeric(trees_df["tree_dbh"], errors="coerce")
trees_df["latitude"] = pd.to_numeric(trees_df["latitude"], errors="coerce")
trees_df["longitude"] = pd.to_numeric(trees_df["longitude"], errors="coerce")

print(f"Live street trees fetched: {len(trees_df)}")
print(f"Mean DBH (inches): {trees_df['tree_dbh'].mean():.1f}")
print()
display(trees_df.head(3))
print()
print("Null counts:")
print(trees_df[["latitude", "longitude", "tree_dbh", "nta"]].isna().sum())

trees_csv = os.path.join(OUTPUT_FOLDER, "data", "street_trees.csv")
trees_df.to_csv(trees_csv, index=False)
print(f"\nSaved: {trees_csv}")

### NYC Wi-Fi Hotspot Locations

LinkNYC kiosks, public library wi-fi, parks wi-fi, and a handful of other providers. The dataset is maintained by DoITT (now the Office of Technology and Innovation). It is a meaningful but incomplete signal of **digital infrastructure** — it omits residential broadband subscriptions entirely, and it overcounts in tourist-heavy zones where commercial hotspots cluster.

In [ ]:
# Wi-Fi Hotspots.
WIFI_COLUMNS = ["ssid", "provider", "type", "borough",
                "ntacode", "ntaname", "latitude", "longitude"]
wifi_df = fetch_nyc_open_data(
    "yjub-udmw.json",
    params={"$select": ",".join(WIFI_COLUMNS)},
    max_rows=20000,
)
wifi_df["latitude"] = pd.to_numeric(wifi_df["latitude"], errors="coerce")
wifi_df["longitude"] = pd.to_numeric(wifi_df["longitude"], errors="coerce")

print(f"Wi-Fi hotspots: {len(wifi_df)}")
print(f"Providers: {wifi_df['provider'].value_counts().head().to_dict()}")
print()
display(wifi_df.head(3))

wifi_csv = os.path.join(OUTPUT_FOLDER, "data", "wifi_hotspots.csv")
wifi_df.to_csv(wifi_csv, index=False)
print(f"\nSaved: {wifi_csv}")

### 311 Service Requests (recent, filtered)

The most-used dataset on NYC Open Data. We pull a recent window and filter to a focused list of housing-condition and environmental complaint types. Remember: **311 is a map of who calls, not where conditions are worst.** We will use it twice in this notebook — once as a stress signal (Module 5 Index C) and once as a civic-engagement signal (Module 5 Index D and Module 4 emergent boundaries). The same data, two different stories.

In [ ]:
# 311 Service Requests — last ~6 months, focal complaint types.
from datetime import datetime, timedelta

cutoff_date = (datetime.utcnow() - timedelta(days=180)).strftime("%Y-%m-%dT00:00:00")

FOCAL_COMPLAINTS = (
    "HEAT/HOT WATER", "UNSANITARY CONDITION", "NOISE - RESIDENTIAL",
    "PAINT/PLASTER", "PLUMBING", "RODENT", "AIR QUALITY",
    "Damaged Tree", "Overgrown Tree/Branches",
)
complaints_clause = ",".join(f"'{c}'" for c in FOCAL_COMPLAINTS)
where_clause = (f"created_date > '{cutoff_date}' "
                f"AND complaint_type IN ({complaints_clause}) "
                f"AND latitude IS NOT NULL")

calls_df = fetch_nyc_open_data(
    "erm2-nwe9.json",
    params={"$select": "unique_key,created_date,complaint_type,borough,"
                       "incident_zip,latitude,longitude",
            "$where": where_clause},
    max_rows=80000,
)
calls_df["latitude"] = pd.to_numeric(calls_df["latitude"], errors="coerce")
calls_df["longitude"] = pd.to_numeric(calls_df["longitude"], errors="coerce")

# Housing vs. environmental subsets for the rate calculations later.
HOUSING_COMPLAINTS = {"HEAT/HOT WATER", "UNSANITARY CONDITION",
                      "PAINT/PLASTER", "PLUMBING", "RODENT"}
ENV_COMPLAINTS = {"AIR QUALITY", "Damaged Tree", "Overgrown Tree/Branches"}
calls_df["complaint_family"] = calls_df["complaint_type"].apply(
    lambda c: "housing" if c in HOUSING_COMPLAINTS
    else ("environmental" if c in ENV_COMPLAINTS else "other"))

print(f"311 calls fetched (last 180 days, focal types): {len(calls_df)}")
print(f"Top complaint types:")
print(calls_df["complaint_type"].value_counts().head(10).to_string())

calls_csv = os.path.join(OUTPUT_FOLDER, "data", "service_calls_311.csv")
calls_df.to_csv(calls_csv, index=False)
print(f"\nSaved: {calls_csv}")

### Heat Vulnerability Index (DOHMH)

Pre-computed by the NYC Department of Health and Mental Hygiene. **The current published version is at ZIP Code Tabulation Area (ZCTA) scale, not NTA** — the dataset schema you fetch below has two columns: `zcta20` and `hvi`. We will not use this as a building block (it would require a ZCTA-to-NTA crosswalk that introduces its own aggregation error); we keep it as a **reference layer** to compare against the indexes we build ourselves.

In [ ]:
# Heat Vulnerability Index. The endpoint sometimes returns 404 if the dataset
# has been re-released under a new ID; if so we surface that as plain English.
hvi_df = fetch_nyc_open_data("4mhf-duep.json", params=None, max_rows=300)

if hvi_df.empty:
    print("HVI dataset returned no rows. The endpoint may have changed.")
    print("Continue with the rest of the notebook — HVI is a comparison-only layer.")
else:
    print(f"HVI rows: {len(hvi_df)}")
    display(hvi_df.head(3))
    hvi_csv = os.path.join(OUTPUT_FOLDER, "data", "heat_vulnerability_index.csv")
    hvi_df.to_csv(hvi_csv, index=False)
    print(f"\nSaved: {hvi_csv}")

### Subway Station Entrances and ADA Compliance

Entrance-level data lets us compute walking-distance access rather than station-level access (one station can have many entrances). The MTA publishes two relevant datasets on **NY State Open Data** (not NYC Open Data): one for entrances (lat/lng of every entrance) and one for stations (which carries the `ada` flag). We pull both and join the ADA field from station onto entrance via the shared `station_id`.

The `ada` field is the headline indicator but can lag actual elevator status — recently installed elevators may not be reflected, and existing elevators may be out of service on the day you read this.

In [ ]:
# MTA Subway Entrances and Exits (NY State Open Data). Station-level ADA
# flag lives on the companion stations dataset; we join the two on station_id.
entrances_df = fetch_nyc_open_data(
    "i9wp-a4ja.json",
    params={"$select": "station_id,complex_id,stop_name,division,line,borough,"
                       "entrance_type,entrance_latitude,entrance_longitude"},
    max_rows=5000,
    domain="data.ny.gov",
)
stations_df = fetch_nyc_open_data(
    "39hk-dx4f.json",
    params={"$select": "station_id,stop_name,ada,ada_northbound,ada_southbound"},
    max_rows=1000,
    domain="data.ny.gov",
)

if entrances_df.empty:
    print("Subway entrance dataset returned no rows. Skipping.")
    subway_df = pd.DataFrame()
else:
    # The lat/lng columns from this dataset are entrance_latitude/longitude;
    # normalize to the same column names every other dataset uses.
    entrances_df["latitude"]  = pd.to_numeric(entrances_df["entrance_latitude"],
                                              errors="coerce")
    entrances_df["longitude"] = pd.to_numeric(entrances_df["entrance_longitude"],
                                              errors="coerce")
    # Bring ADA in from the station dataset.
    if not stations_df.empty and "station_id" in entrances_df.columns:
        ada_lookup = stations_df[["station_id", "ada"]].copy()
        subway_df = entrances_df.merge(ada_lookup, on="station_id", how="left")
    else:
        subway_df = entrances_df.copy()
        subway_df["ada"] = None

    print(f"Subway entrances : {len(subway_df)}")
    if "ada" in subway_df.columns:
        ada_yes = (subway_df["ada"].astype(str).str.strip() == "1").sum()
        print(f"ADA accessible   : {ada_yes} ({100*ada_yes/len(subway_df):.1f}%)")
    display(subway_df.head(3))
    subway_csv = os.path.join(OUTPUT_FOLDER, "data", "subway_entrances.csv")
    subway_df.to_csv(subway_csv, index=False)
    print(f"\nSaved: {subway_csv}")

---

## Module 3 — Spatial Join and Geographic Aggregation

The act of aggregating is itself an analytical choice. When we ask "how many trees does this neighborhood have?", the answer depends on **how we drew the neighborhood**. The same five blocks, aggregated to two different geographies, can produce two different rankings.

In this module we will:
1. Load the tract and NTA boundary GeoJSONs from Module 1
2. Spatially join every point dataset (trees, wi-fi, subway entrances, 311 calls) to **both** geographies
3. Compute the same aggregated variables at each level
4. Display a comparison table so you can see how the numbers shift

**Module 3 depends on:** Modules 0, 1, 2.

In [ ]:
# Module 3 — spatial joins + aggregation at tract and NTA scale.
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load boundary layers from disk so this module is independently re-runnable.
tracts_gdf = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data", "nyc_tracts.geojson"))
ntas_gdf   = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data", "nyc_ntas.geojson"))
tracts_gdf = tracts_gdf.to_crs(epsg=4326)
ntas_gdf   = ntas_gdf.to_crs(epsg=4326)

# Resolve the NTA name column across schema versions (2010 vs 2020 datasets).
if "ntaname" in ntas_gdf.columns:
    ntas_gdf["nta_name"] = ntas_gdf["ntaname"]
elif "nta2020_name" in ntas_gdf.columns:
    ntas_gdf["nta_name"] = ntas_gdf["nta2020_name"]
elif "ntaname" not in ntas_gdf.columns and "name" in ntas_gdf.columns:
    ntas_gdf["nta_name"] = ntas_gdf["name"]

def points_from_lat_lng(dataframe, lat_col="latitude", lng_col="longitude"):
    """Return a GeoDataFrame keyed on the same index with WGS84 point geometry."""
    valid = dataframe.dropna(subset=[lat_col, lng_col]).copy()
    valid["geometry"] = [Point(lng, lat) for lng, lat
                         in zip(valid[lng_col], valid[lat_col])]
    return gpd.GeoDataFrame(valid, geometry="geometry", crs="EPSG:4326")

# Load Module 2 outputs from disk so order of execution is forgiving.
trees_df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "street_trees.csv"))
wifi_df  = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "wifi_hotspots.csv"))
calls_df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "service_calls_311.csv"))
pluto_df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "pluto_sample.csv"))
subway_path = os.path.join(OUTPUT_FOLDER, "data", "subway_entrances.csv")
subway_df = pd.read_csv(subway_path) if os.path.exists(subway_path) else pd.DataFrame()

trees_gdf  = points_from_lat_lng(trees_df)
wifi_gdf   = points_from_lat_lng(wifi_df)
calls_gdf  = points_from_lat_lng(calls_df)
pluto_gdf  = points_from_lat_lng(pluto_df)
subway_gdf = (points_from_lat_lng(subway_df) if not subway_df.empty
              else gpd.GeoDataFrame(columns=["geometry"], geometry="geometry",
                                    crs="EPSG:4326"))

print(f"Tree points : {len(trees_gdf)}")
print(f"Wi-Fi points: {len(wifi_gdf)}")
print(f"311 points  : {len(calls_gdf)}")
print(f"PLUTO points: {len(pluto_gdf)}")
print(f"Subway pts  : {len(subway_gdf)}")

In [ ]:
# Aggregation function: compute one row per polygon with all derived variables.
def aggregate_to_polygons(polys_gdf, name_col, acs_estimates_gdf=None):
    """Return one row per polygon with derived variables and population."""
    polys = polys_gdf.copy()
    polys["poly_index"] = range(len(polys))

    def sjoin_count(point_gdf, where=None):
        if point_gdf.empty:
            return pd.Series([0] * len(polys), index=polys.index)
        gdf = point_gdf if where is None else point_gdf[where]
        if gdf.empty:
            return pd.Series([0] * len(polys), index=polys.index)
        joined = gpd.sjoin(gdf, polys[["poly_index", "geometry"]],
                           how="left", predicate="within")
        counts = joined.groupby("poly_index").size()
        return counts.reindex(polys["poly_index"], fill_value=0).values

    polys["tree_count"]    = sjoin_count(trees_gdf)
    polys["wifi_count"]    = sjoin_count(wifi_gdf)
    polys["subway_count"]  = sjoin_count(subway_gdf)
    polys["calls_total"]   = sjoin_count(calls_gdf)
    polys["calls_housing"] = sjoin_count(
        calls_gdf, calls_gdf["complaint_family"] == "housing")
    polys["calls_env"]     = sjoin_count(
        calls_gdf, calls_gdf["complaint_family"] == "environmental")
    polys["pluto_lots"]    = sjoin_count(pluto_gdf)
    polys["below_grade_lots"] = sjoin_count(
        pluto_gdf, pluto_gdf["below_grade"] == True)

    # Mean tree DBH per polygon as a canopy proxy.
    if not trees_gdf.empty:
        tree_sjoin = gpd.sjoin(trees_gdf, polys[["poly_index", "geometry"]],
                               how="left", predicate="within")
        mean_dbh = tree_sjoin.groupby("poly_index")["tree_dbh"].mean()
        polys["canopy_proxy"] = mean_dbh.reindex(polys["poly_index"], fill_value=0).values
    else:
        polys["canopy_proxy"] = 0.0

    # ADA share of nearby subway entrances. The MTA dataset encodes ADA as
    # a string "1" / "0"; older NYC OD versions used "TRUE" / "FALSE".
    if not subway_gdf.empty and "ada" in subway_gdf.columns:
        ada_string = subway_gdf["ada"].astype(str).str.strip().str.upper()
        ada_mask = ada_string.isin({"1", "TRUE", "Y", "YES"})
        polys["subway_ada_count"] = sjoin_count(subway_gdf, ada_mask)
        polys["ada_subway_pct"] = np.where(
            polys["subway_count"] > 0,
            100 * polys["subway_ada_count"] / polys["subway_count"], np.nan)
    else:
        polys["subway_ada_count"] = 0
        polys["ada_subway_pct"]   = np.nan

    polys["below_grade_pct"] = np.where(
        polys["pluto_lots"] > 0,
        100 * polys["below_grade_lots"] / polys["pluto_lots"], np.nan)

    # Lot area per polygon for tree density.
    if not pluto_gdf.empty:
        pluto_join = gpd.sjoin(pluto_gdf[["lotarea", "geometry"]],
                               polys[["poly_index", "geometry"]],
                               how="left", predicate="within")
        total_lotarea_sqft = pluto_join.groupby("poly_index")["lotarea"].sum()
        polys["total_lotarea_ha"] = (total_lotarea_sqft.reindex(polys["poly_index"],
                                                                 fill_value=0)
                                     .values * 0.0000092903)  # sqft -> hectares
    else:
        polys["total_lotarea_ha"] = 0.0
    polys["tree_density"] = np.where(
        polys["total_lotarea_ha"] > 0,
        polys["tree_count"] / polys["total_lotarea_ha"], 0.0)

    return polys

# Aggregate to tract scale, attaching ACS population so we can normalize rates.
acs_lookup = acs_gdf.drop(columns="geometry").copy()
tracts_with_acs = tracts_gdf.merge(acs_lookup, on="GEOID", how="left")
tract_agg = aggregate_to_polygons(tracts_with_acs, name_col="GEOID")

# Per-1,000-resident rates that depend on ACS population.
population = pd.to_numeric(tract_agg.get("B01001_001E"), errors="coerce")
tract_agg["wifi_density"]            = np.where(population > 0,
                                                1000 * tract_agg["wifi_count"] / population, np.nan)
tract_agg["housing_311_rate"]        = np.where(population > 0,
                                                1000 * tract_agg["calls_housing"] / population, np.nan)
tract_agg["environmental_311_rate"]  = np.where(population > 0,
                                                1000 * tract_agg["calls_env"] / population, np.nan)
tract_agg["rent_burden_pct"]         = pd.to_numeric(tract_agg.get("B25071_001E"), errors="coerce")
tract_agg["transit_dependency"]      = np.where(
    pd.to_numeric(tract_agg.get("B08301_001E"), errors="coerce") > 0,
    pd.to_numeric(tract_agg.get("B08301_010E"), errors="coerce") /
    pd.to_numeric(tract_agg.get("B08301_001E"), errors="coerce"), np.nan)
tract_agg["poverty_rate"]            = np.where(population > 0,
                                                pd.to_numeric(tract_agg.get("B17001_002E"), errors="coerce") / population, np.nan)
tract_agg["vacant_unit_pct"]         = np.where(population > 0,
                                                100 * pd.to_numeric(tract_agg.get("B25002_003E"), errors="coerce") / population, np.nan)

# Aggregate to NTA scale by first spatially joining tract centroids to NTAs.
tract_centroids = tract_agg.copy()
tract_centroids["geometry"] = tract_centroids.geometry.centroid
tract_to_nta = gpd.sjoin(tract_centroids, ntas_gdf[["nta_name", "geometry"]],
                         how="left", predicate="within")

# Mean of derived rates / population-weighted sum where appropriate.
def _safe_weighted_mean(values, weights):
    """Population-weighted mean that tolerates NaN values and weights."""
    mask = pd.notna(values) & pd.notna(weights) & (weights > 0)
    if not mask.any():
        return np.nan
    return float((values[mask] * weights[mask]).sum() / weights[mask].sum())

nta_rows = []
for nta_name, group in tract_to_nta.groupby("nta_name"):
    pop = pd.to_numeric(group.get("B01001_001E"), errors="coerce")
    nta_rows.append({
        "nta_name": nta_name,
        "tree_density":           float(group["tree_density"].mean()),
        "canopy_proxy":           float(group["canopy_proxy"].mean()),
        "wifi_density":           _safe_weighted_mean(group["wifi_density"], pop),
        "subway_access":          int(group["subway_count"].sum()),
        "ada_subway_pct":         float(group["ada_subway_pct"].mean(skipna=True)),
        "below_grade_pct":        float(group["below_grade_pct"].mean(skipna=True)),
        "housing_311_rate":       _safe_weighted_mean(group["housing_311_rate"], pop),
        "environmental_311_rate": _safe_weighted_mean(group["environmental_311_rate"], pop),
        "rent_burden_pct":        _safe_weighted_mean(group["rent_burden_pct"], pop),
        "transit_dependency":     _safe_weighted_mean(group["transit_dependency"], pop),
        "poverty_rate":           _safe_weighted_mean(group["poverty_rate"], pop),
        "vacant_unit_pct":        _safe_weighted_mean(group["vacant_unit_pct"], pop),
        "population":             float(pop.sum(skipna=True)),
    })
nta_agg = ntas_gdf.merge(pd.DataFrame(nta_rows), on="nta_name", how="left")

# Subway access at tract scale uses entrance count within tract; for tract-level
# walking access we approximate by counting subway entrances within 400 m of
# the tract centroid (re-project to NY State Plane in feet for buffering).
print("Computing 400 m walking-distance subway access at tract scale...")
tract_proj = tract_agg.to_crs(epsg=2263)  # NY Long Island, in feet
buffer_ft = 400 * 3.28084  # 400 m in feet
tract_proj["geometry"] = tract_proj.geometry.centroid.buffer(buffer_ft)
if not subway_gdf.empty:
    subway_proj = subway_gdf.to_crs(epsg=2263)
    near_subway = gpd.sjoin(subway_proj[["geometry"]],
                            tract_proj[["GEOID", "geometry"]],
                            how="left", predicate="within")
    walk_access = near_subway.groupby("GEOID").size()
    tract_agg["subway_access"] = walk_access.reindex(tract_agg["GEOID"],
                                                     fill_value=0).values
else:
    tract_agg["subway_access"] = 0

print(f"Tract-level aggregated rows : {len(tract_agg)}")
print(f"NTA-level aggregated rows   : {len(nta_agg)}")

# Save outputs.
tract_out = os.path.join(OUTPUT_FOLDER, "data", "tract_aggregated.geojson")
nta_out   = os.path.join(OUTPUT_FOLDER, "data", "nta_aggregated.geojson")
tract_agg.to_file(tract_out, driver="GeoJSON")
nta_agg.to_file(nta_out, driver="GeoJSON")
print(f"\nSaved: {tract_out}")
print(f"Saved: {nta_out}")

In [ ]:
# Comparison table: the same five neighborhoods at tract and NTA scale.
import pandas as pd
SAMPLE_NTAS = ["East Harlem (North)", "East Harlem (South)",
               "Bedford-Stuyvesant (West)", "Astoria (Central)",
               "Brownsville"]
# Fall back to whatever five NTAs exist if the named ones aren't in this dataset.
matched = nta_agg[nta_agg["nta_name"].isin(SAMPLE_NTAS)]["nta_name"].tolist()
if len(matched) < 5:
    matched = nta_agg["nta_name"].dropna().head(5).tolist()

nta_sample = nta_agg[nta_agg["nta_name"].isin(matched)][
    ["nta_name", "tree_density", "canopy_proxy", "wifi_density",
     "below_grade_pct", "housing_311_rate", "rent_burden_pct"]
].copy()
nta_sample.insert(1, "geography", "NTA")

# Pull matching tracts (one row per tract that falls inside one of the sample NTAs).
tract_sample_rows = tract_to_nta[tract_to_nta["nta_name"].isin(matched)][
    ["GEOID", "nta_name", "tree_density", "canopy_proxy", "wifi_density",
     "below_grade_pct", "housing_311_rate", "rent_burden_pct"]
].copy()

# Average within each NTA for a clean comparison.
tract_summary = (tract_sample_rows
                 .groupby("nta_name")
                 [["tree_density", "canopy_proxy", "wifi_density",
                   "below_grade_pct", "housing_311_rate", "rent_burden_pct"]]
                 .mean().reset_index())
tract_summary.insert(1, "geography", "Tract (avg)")

comparison = pd.concat([nta_sample, tract_summary], ignore_index=True)
comparison = comparison.sort_values(["nta_name", "geography"])
print("Same five neighborhoods, two geographies:")
display(comparison.round(2))

**Notice how the numbers shift between geographies.** A tract-level average of housing 311 rate within an NTA can look very different from the NTA-level rate computed against NTA population — the denominator changes, and so do the visible peaks. Which scale reveals more depends on what you are trying to see. **Aggregate up too far and you lose the block where the problem is. Aggregate down too far and the margins of error swallow the signal.**

---

## Module 4 — Emergent Boundaries from 311 Data

This module is framed as an **experiment**, not a method.

**What it does.** Run K-Means clustering on the lat/lng coordinates of 311 service requests and draw the convex hull of each cluster. The result is a kind of pseudo-neighborhood map derived from where complaints physically concentrate, rather than from the lines that the Department of City Planning drew.

**What it conceals.** 311 data reflects who picks up the phone (or opens the app), not just where conditions are worst. Wealthier, more digitally literate neighborhoods call 311 more frequently per capita. English fluency, trust in city agencies, and fear of landlord retaliation all dampen call rates among the most vulnerable households. **The boundaries you are about to draw are partly a map of civic engagement and internet access, not just of urban conditions.**

The NYT's [Neighborhood Boundaries project](https://www.nytimes.com/interactive/2023/upshot/analog-neighborhoods.html) approached this question by *asking people directly* — surveying residents about where their neighborhood ended. That is the methodological standard. We are working from administrative records, which is a different and weaker kind of evidence. Keep the gap in mind when you read the map.

**Module 4 depends on:** Modules 0, 1, 2.

In [ ]:
# Module 4 — emergent boundaries from 311 clusters.
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import MultiPoint, Polygon
from sklearn.cluster import KMeans
import ipywidgets as widgets
from IPython.display import display

# Reload 311 data so this module is independently runnable.
calls_df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "service_calls_311.csv"))
calls_df = calls_df.dropna(subset=["latitude", "longitude"]).copy()
ntas_gdf = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data", "nyc_ntas.geojson"))
ntas_gdf = ntas_gdf.to_crs(epsg=4326)

available_complaints = sorted(calls_df["complaint_type"].unique().tolist())

filter_widget = widgets.Dropdown(
    options=["(all focal complaint types)"] + available_complaints,
    value="(all focal complaint types)",
    description="Filter:",
    style={"description_width": "initial"})
k_widget = widgets.IntSlider(value=40, min=10, max=100, step=5,
                             description="Clusters K:",
                             style={"description_width": "initial"})
run_button = widgets.Button(description="Run clustering",
                            button_style="primary")
output_area = widgets.Output()

display(widgets.VBox([filter_widget, k_widget, run_button, output_area]))

def _on_run_clicked(_):
    with output_area:
        output_area.clear_output()
        selection = filter_widget.value
        k_clusters = k_widget.value
        working = calls_df.copy()
        if selection != "(all focal complaint types)":
            working = working[working["complaint_type"] == selection]
        if len(working) < k_clusters:
            print(f"Only {len(working)} points available — reduce K below that.")
            return

        coords = working[["latitude", "longitude"]].to_numpy()
        kmeans = KMeans(n_clusters=k_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(coords)
        working["cluster"] = labels

        # Build convex hull polygons per cluster.
        hulls = []
        for cluster_id, group in working.groupby("cluster"):
            if len(group) < 3:
                continue
            hull_geometry = MultiPoint(
                [(lng, lat) for lat, lng in
                 zip(group["latitude"], group["longitude"])]).convex_hull
            if not isinstance(hull_geometry, Polygon):
                continue
            dominant = group["complaint_type"].mode().iloc[0]
            hulls.append({"cluster": int(cluster_id),
                          "n_points": int(len(group)),
                          "dominant_complaint": dominant,
                          "geometry": hull_geometry})
        hulls_gdf = gpd.GeoDataFrame(hulls, crs="EPSG:4326")

        # Color hulls by dominant complaint via a stable palette.
        palette = ["#1B1B33", "#3C4ED6", "#16A085", "#E67E22", "#C0392B",
                   "#9B59B6", "#27AE60", "#D4AC0D", "#34495E", "#7B241C",
                   "#117864", "#212F3D"]
        unique_complaints = hulls_gdf["dominant_complaint"].unique().tolist()
        color_lookup = {complaint: palette[i % len(palette)]
                        for i, complaint in enumerate(unique_complaints)}

        # Folium map: NTA outlines in cobalt, cluster hulls in their complaint color.
        center = [working["latitude"].mean(), working["longitude"].mean()]
        cluster_map = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")

        folium.GeoJson(
            ntas_gdf,
            style_function=lambda feature: {"color": "#3C4ED6",
                                            "weight": 1.2,
                                            "fillOpacity": 0},
            name="NTA boundaries",
        ).add_to(cluster_map)

        for _, row in hulls_gdf.iterrows():
            cluster_color = color_lookup[row["dominant_complaint"]]
            folium.GeoJson(
                row["geometry"].__geo_interface__,
                style_function=lambda feature, color=cluster_color: {
                    "fillColor": color, "color": color,
                    "weight": 1, "fillOpacity": 0.45},
                tooltip=(f"Cluster {row['cluster']}<br>"
                         f"Points: {row['n_points']}<br>"
                         f"Dominant: {row['dominant_complaint']}"),
            ).add_to(cluster_map)

        folium.LayerControl().add_to(cluster_map)
        display(cluster_map)

        out_path = os.path.join(OUTPUT_FOLDER, "data", "emergent_boundaries.geojson")
        hulls_gdf.to_file(out_path, driver="GeoJSON")
        print(f"Saved cluster polygons: {out_path}")
        print(f"Filter: {selection} | K = {k_clusters} | "
              f"{len(hulls_gdf)} valid hulls drawn")

run_button.on_click(_on_run_clicked)

**Compare these boundaries to the NTA map.** Where do they align? Where do they diverge? What might explain the divergence?

Now ask the harder question: **whose experience is encoded in this map, and whose is missing?** A cluster of 311 calls is a sign that someone in that area believed a 311 call would be useful — that the city would respond, that they would not face retaliation, that they could navigate the system in their language and on their device. The absence of clusters in some neighborhoods is not evidence that things are fine. It is evidence that the city has not been called.

If you want to take this further: layer this map against the Heat Vulnerability Index NTA map and look for places where they disagree. Those are the places where the bottom-up signal of 311 most clearly fails to match the top-down measurement of harm.

---

## Module 5 — Example Indexes

Four worked examples. Each one has an opinion about what matters in a city. Each one is wrong about some neighborhoods and right about others. Together, they will give you a sense of what an index *can* and *cannot* do.

For each index you will see:
- An introduction explaining what the index is trying to measure and what it cannot capture
- The variable selection rationale
- The normalization method, in plain English
- The weighting scheme, with what the weights *assert* about relative importance
- A choropleth map at NTA level
- A horizontal bar chart of the top 10 and bottom 10 scoring areas
- A note on what the index misses

**Module 5 depends on:** Modules 0-3.

In [ ]:
# Shared scoring infrastructure used by every index in this module.
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

nta_agg = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data", "nta_aggregated.geojson"))
nta_agg = nta_agg.to_crs(epsg=4326)

def normalize_min_max(series):
    """Map values to [0, 1] using the observed min and max."""
    minimum = series.min(skipna=True)
    maximum = series.max(skipna=True)
    if maximum == minimum:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - minimum) / (maximum - minimum)

def normalize_percentile(series):
    """Map values to [0, 1] by their rank percentile."""
    return series.rank(pct=True, method="average")

def normalize_zscore(series):
    """Center to mean 0, scale to unit variance, then clip to a 0-1 window."""
    mu = series.mean(skipna=True)
    sigma = series.std(skipna=True)
    if sigma == 0 or pd.isna(sigma):
        return pd.Series(np.zeros(len(series)), index=series.index)
    z = (series - mu) / sigma
    # Map z in [-3, 3] to [0, 1] for choropleth display.
    return ((z.clip(-3, 3) + 3) / 6)

NORMALIZERS = {
    "min_max":    normalize_min_max,
    "percentile": normalize_percentile,
    "zscore":     normalize_zscore,
}

def compute_index(gdf, components, normalizer="min_max"):
    """Return a Series of weighted, direction-corrected, normalized scores.

    components : list of dicts with keys: variable, weight, invert (bool).
    """
    normalizer_fn = NORMALIZERS[normalizer]
    score = pd.Series(np.zeros(len(gdf)), index=gdf.index)
    for component in components:
        column = component["variable"]
        weight = float(component["weight"])
        invert = bool(component.get("invert", False))
        values = pd.to_numeric(gdf[column], errors="coerce")
        normalized = normalizer_fn(values).fillna(0.5)
        if invert:
            normalized = 1.0 - normalized
        score = score + weight * normalized
    return score

def draw_choropleth(gdf, score_column, components, title, map_path):
    """Render a folium choropleth and save to disk."""
    center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
    choropleth_map = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")
    tooltip_fields = ["nta_name", score_column] + [c["variable"] for c in components]
    tooltip_aliases = ["NTA", "Score"] + [c["variable"] for c in components]
    folium.Choropleth(
        geo_data=gdf.__geo_interface__,
        data=gdf,
        columns=["nta_name", score_column],
        key_on="feature.properties.nta_name",
        fill_color="YlGnBu",
        fill_opacity=0.8,
        line_opacity=0.2,
        legend_name=title,
    ).add_to(choropleth_map)
    folium.GeoJson(
        gdf,
        style_function=lambda feature: {"fillOpacity": 0, "color": "#333", "weight": 0.3},
        tooltip=folium.features.GeoJsonTooltip(fields=tooltip_fields,
                                               aliases=tooltip_aliases,
                                               localize=True),
    ).add_to(choropleth_map)
    choropleth_map.save(map_path)
    return choropleth_map

def draw_top_bottom_bars(gdf, score_column, title, n=10):
    """Render a top-n / bottom-n horizontal bar chart inline."""
    sorted_gdf = gdf.dropna(subset=[score_column]).sort_values(score_column, ascending=False)
    top_rows    = sorted_gdf.head(n)
    bottom_rows = sorted_gdf.tail(n).iloc[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    axes[0].barh(top_rows["nta_name"], top_rows[score_column], color="#3C4ED6")
    axes[0].set_title(f"Top {n} — {title}")
    axes[0].invert_yaxis()
    axes[1].barh(bottom_rows["nta_name"], bottom_rows[score_column], color="#C0392B")
    axes[1].set_title(f"Bottom {n} — {title}")
    axes[1].invert_yaxis()
    plt.tight_layout()
    plt.show()

print("Shared scoring infrastructure ready.")

### Index A — Green Equity Index

**What it measures.** Access to tree canopy and green infrastructure, weighted by the populations most exposed to heat and most likely to bear the burden of green-infrastructure gaps.

**Variables and weights.**
- `tree_density` (0.25) — more trees per hectare of lot area
- `canopy_proxy` (0.20) — larger mean tree diameter as a coverage proxy
- `below_grade_pct` (0.20, **inverted**) — more below-grade buildings = more flood vulnerability = lower score
- `poverty_rate` (0.20, **inverted**) — higher poverty = lower score, surfacing where green-infrastructure gaps compound with economic disadvantage
- `environmental_311_rate` (0.15, **inverted**) — more tree complaints = more unmet need = lower score

**Normalization: min-max scaling.** All values mapped to 0-1 using the observed min and max in NYC. Simple to read but **sensitive to outliers** — a single very-high-canopy NTA can pull everyone else's score down.

**What this index will not tell you.** Park access (Street Tree Census is *streets only*). Surface temperature from the urban heat island. Quality of green space (a small grove of healthy oaks scores the same as the same number of sickly ginkgos). Maintenance regimes, which determine whether a tree-dense block actually feels green or feels like a strip of asphalt with sticks in it.

In [ ]:
# Index A — Green Equity Index.
green_components = [
    {"variable": "tree_density",           "weight": 0.25, "invert": False},
    {"variable": "canopy_proxy",           "weight": 0.20, "invert": False},
    {"variable": "below_grade_pct",        "weight": 0.20, "invert": True},
    {"variable": "poverty_rate",           "weight": 0.20, "invert": True},
    {"variable": "environmental_311_rate", "weight": 0.15, "invert": True},
]

nta_agg["green_equity_score"] = compute_index(nta_agg, green_components,
                                              normalizer="min_max")
map_path = os.path.join(OUTPUT_FOLDER, "maps", "index_a_green_equity.html")
green_map = draw_choropleth(nta_agg, "green_equity_score", green_components,
                            "Green Equity Index", map_path)
display(green_map)
draw_top_bottom_bars(nta_agg, "green_equity_score", "Green Equity Index", n=10)
print(f"Saved map: {map_path}")

### Index B — Transit and Digital Equity Index

**What it measures.** Combined access to physical mobility (subway entrances within walking distance, with an ADA adjustment) and digital mobility (public wi-fi density), weighted against the share of residents who depend on transit.

**Variables and weights.**
- `subway_access` (0.30) — entrances within 400 m
- `ada_subway_pct` (0.25) — share of nearby entrances that are ADA accessible
- `wifi_density` (0.25) — public hotspots per 1,000 residents
- `transit_dependency` (0.20) — share of commuters using public transit, so high dependency + low access reads as inequity

**Normalization: percentile rank.** Each NTA is ranked relative to every other NTA. More robust to outliers than min-max. The tradeoff: percentile rank tells you *where you stand* but not *how far apart* you are — the top NTA and the second-from-top can be either nearly tied or wildly separated, and you cannot see the difference from the rank alone.

**What this index will not tell you.** Home broadband subscription rates (ACS B28002 would help, but with very high MOE at tract scale). Quality of service — frequency of trains, latency of wi-fi, station overcrowding. The walking distance buffer is geometric, not topological — a subway entrance 350m away across a highway scores the same as one 350m away across a sidewalk.

In [ ]:
# Index B — Transit and Digital Equity Index.
transit_components = [
    {"variable": "subway_access",      "weight": 0.30, "invert": False},
    {"variable": "ada_subway_pct",     "weight": 0.25, "invert": False},
    {"variable": "wifi_density",       "weight": 0.25, "invert": False},
    {"variable": "transit_dependency", "weight": 0.20, "invert": False},
]
nta_agg["transit_digital_score"] = compute_index(nta_agg, transit_components,
                                                 normalizer="percentile")
map_path = os.path.join(OUTPUT_FOLDER, "maps", "index_b_transit_digital.html")
transit_map = draw_choropleth(nta_agg, "transit_digital_score", transit_components,
                              "Transit and Digital Equity Index", map_path)
display(transit_map)
draw_top_bottom_bars(nta_agg, "transit_digital_score",
                     "Transit and Digital Equity Index", n=10)
print(f"Saved map: {map_path}")

### Index C — Housing Stress Index

**What it measures.** Concentration of housing-instability signals.

**Variables and weights.**
- `rent_burden_pct` (0.35) — median gross rent as percentage of household income
- `poverty_rate` (0.25)
- `housing_311_rate` (0.25) — complaints about heat, pests, plumbing, conditions
- `vacant_unit_pct` (0.15) — vacancy as a signal of disinvestment or speculation

**Normalization: z-score.** Each variable is centered on its mean and scaled by its standard deviation. Z-scores are good at surfacing **extremes** — they pull outliers visible in a way that min-max would compress. They are bad at intuitive interpretation: most readers do not think in standard deviations.

**What this index will not tell you.** The most distressed households are often least likely to call 311 — fear of landlord retaliation, language barriers, distrust of city agencies. The `housing_311_rate` variable in this index almost certainly *undercounts* distress in the most vulnerable buildings. Other gaps: short-term displacement (people priced out before the year-over-year ACS catches up), illegal subdivisions (not in vacancy data because they are not legally vacant), informal housing arrangements (couch-surfing, doubled-up families).

In [ ]:
# Index C — Housing Stress Index.
housing_components = [
    {"variable": "rent_burden_pct",  "weight": 0.35, "invert": False},
    {"variable": "poverty_rate",     "weight": 0.25, "invert": False},
    {"variable": "housing_311_rate", "weight": 0.25, "invert": False},
    {"variable": "vacant_unit_pct",  "weight": 0.15, "invert": False},
]
nta_agg["housing_stress_score"] = compute_index(nta_agg, housing_components,
                                                normalizer="zscore")
map_path = os.path.join(OUTPUT_FOLDER, "maps", "index_c_housing_stress.html")
housing_map = draw_choropleth(nta_agg, "housing_stress_score", housing_components,
                              "Housing Stress Index", map_path)
display(housing_map)
draw_top_bottom_bars(nta_agg, "housing_stress_score", "Housing Stress Index", n=10)
print(f"Saved map: {map_path}")

### Index D — Neighborhood Cohesion Proxy Index

This is the most speculative index in the notebook.

In 2023 the NYT's [Neighborhood Boundaries project](https://www.nytimes.com/interactive/2023/upshot/analog-neighborhoods.html) used survey data — asking real residents — to measure the degree to which people in a given area *agree* on where their neighborhood begins and ends. High agreement reads as shared identity and social cohesion. Low agreement reads as fragmentation. That kind of data is not available to us through public APIs.

What we can do, transparently, is **approximate** cohesion with a set of open-data proxies and be honest about the gap. The proxies here:
- Civic engagement (311 calls used as a **floor**, not a ceiling — more calls indicates more residents who believe their voice will be heard)
- Structural stability (low below-grade share, low vacancy)
- Shared infrastructure (transit dependency, public wi-fi density)
- Tenure security (low rent burden as a proxy for low displacement pressure)
- Shared public realm (tree density as a proxy for quality of street life)

**Variables and weights.**
- `housing_311_rate` (0.20) — engagement floor, **not** distress here
- `below_grade_pct` (0.10, **inverted**) — structural stability proxy
- `transit_dependency` (0.15) — shared infrastructure use
- `tree_density` (0.20) — quality of shared public realm
- `rent_burden_pct` (0.20, **inverted**) — displacement pressure
- `wifi_density` (0.15) — digital infrastructure as social glue

**Normalization: percentile rank.** Robust to outliers, and ranks are easier to interpret in a "proxy of proxies" context where the absolute values are not directly meaningful.

**What this index does not tell you.** The variables are proxies for proxies. The NYT project asked people directly — we are inferring from city records. The gap between those two methods is the gap between administrative data and lived experience. Read this index for what it is: a **best-effort approximation**, with all of the bias the data carries, and not a substitute for talking to residents.

In [ ]:
# Index D — Neighborhood Cohesion Proxy Index.
cohesion_components = [
    {"variable": "housing_311_rate",   "weight": 0.20, "invert": False},
    {"variable": "below_grade_pct",    "weight": 0.10, "invert": True},
    {"variable": "transit_dependency", "weight": 0.15, "invert": False},
    {"variable": "tree_density",       "weight": 0.20, "invert": False},
    {"variable": "rent_burden_pct",    "weight": 0.20, "invert": True},
    {"variable": "wifi_density",       "weight": 0.15, "invert": False},
]
nta_agg["cohesion_proxy_score"] = compute_index(nta_agg, cohesion_components,
                                                normalizer="percentile")
map_path = os.path.join(OUTPUT_FOLDER, "maps", "index_d_cohesion_proxy.html")
cohesion_map = draw_choropleth(nta_agg, "cohesion_proxy_score", cohesion_components,
                               "Neighborhood Cohesion Proxy Index", map_path)
display(cohesion_map)
draw_top_bottom_bars(nta_agg, "cohesion_proxy_score",
                     "Neighborhood Cohesion Proxy Index", n=10)
print(f"Saved map: {map_path}")

# Persist the example-index scores onto disk so Module 7 can read them.
nta_agg.to_file(os.path.join(OUTPUT_FOLDER, "data", "nta_aggregated.geojson"),
                driver="GeoJSON")

---

## Module 6 — Build Your Own Index

You have seen four indexes. Each one made choices — about what to measure, how to normalize, and how to weight. Now you will make those choices yourself.

**There is no correct index. There is only a more or less legible one. Your job is to be able to defend every decision you make.**

A weight is a statement about what matters more. If you weight tree canopy at 0.4 and broadband access at 0.1, you are saying that green infrastructure is **four times** more important than digital equity. Is that true for the community you are designing for? Who decided?

The NYT Neighborhood Boundaries project earned the right to publish a methodology because it was transparent — about the survey design, the sample, the validation. The methodology was inspectable, so the conclusion was contestable. The team used primary survey data that is not publicly replicable, but the **spirit of transparency** is what we are aiming at here. Every choice you make in this module will end up on a methodology card you can defend.

**Module 6 depends on:** Modules 0-3 (and Module 5 if you want to compare your index to the four examples).

In [ ]:
# Module 6 — interactive index builder.
import os
import json
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# Load whichever geography the student picks at run time, but pre-load both so
# the dropdown can switch instantly.
nta_agg   = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data", "nta_aggregated.geojson")).to_crs(epsg=4326)
tract_agg = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data", "tract_aggregated.geojson")).to_crs(epsg=4326)

# Catalogue of available variables and short descriptions.
VARIABLE_CATALOG = {
    "tree_density":            "Street trees per hectare of lot area",
    "canopy_proxy":            "Mean tree DBH (inches) — proxy for canopy coverage",
    "wifi_density":            "Public wi-fi hotspots per 1,000 residents",
    "subway_access":           "Subway entrances within 400 m of centroid",
    "ada_subway_pct":          "Share of nearby subway entrances that are ADA",
    "below_grade_pct":         "Share of PLUTO lots with below-grade space",
    "housing_311_rate":        "Housing 311 calls per 1,000 residents",
    "environmental_311_rate":  "Environmental 311 calls per 1,000 residents",
    "rent_burden_pct":         "Median gross rent as percent of income (ACS)",
    "transit_dependency":      "Share of commuters using public transit (ACS)",
    "poverty_rate":            "Population below poverty line / total population (ACS)",
    "vacant_unit_pct":         "Vacant housing units / total population (ACS)",
}

NORMALIZER_EXPLANATIONS = {
    "min_max":    "Maps every value to a 0-1 range using the observed minimum and maximum. Easy to read but sensitive to outliers — one extreme NTA can compress everyone else.",
    "percentile": "Ranks each NTA against every other NTA, then converts the rank to a percentile. Robust to outliers but loses information about absolute spacing.",
    "zscore":     "Centers values on their mean and scales by their standard deviation. Good at surfacing extremes but harder to interpret intuitively.",
}

# --- Step 1: variable selection
variable_picker = widgets.SelectMultiple(
    options=[(f"{name} — {desc}", name)
             for name, desc in VARIABLE_CATALOG.items()],
    value=("tree_density", "rent_burden_pct", "transit_dependency"),
    rows=12,
    description="Variables:",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "initial"})

# --- Step 2-4: dynamic controls placeholder
direction_box   = widgets.VBox([])
weight_box      = widgets.VBox([])
rationale_box   = widgets.VBox([])
weight_total_label = widgets.HTML(value="<b>Total weight: 0.00</b>")

direction_toggles = {}
weight_sliders    = {}
rationale_inputs  = {}

# --- Step 3: normalization dropdown
normalization_picker = widgets.Dropdown(
    options=[("Min-Max", "min_max"), ("Percentile Rank", "percentile"),
             ("Z-Score", "zscore")],
    value="min_max", description="Normalization:",
    style={"description_width": "initial"})
normalization_help = widgets.HTML(
    value=f"<i>{NORMALIZER_EXPLANATIONS['min_max']}</i>")

def _on_norm_change(change):
    normalization_help.value = f"<i>{NORMALIZER_EXPLANATIONS[change['new']]}</i>"
normalization_picker.observe(_on_norm_change, names="value")

# --- Step 5: geography dropdown
geography_picker = widgets.Dropdown(
    options=[("NTA", "nta"), ("Census Tract", "tract")],
    value="nta", description="Geography:",
    style={"description_width": "initial"})

# --- Step 6: metadata fields
index_name_input = widgets.Text(value="My Index", description="Index name:",
                                style={"description_width": "initial"},
                                layout=widgets.Layout(width="600px"))
purpose_input    = widgets.Text(value="", placeholder="One-sentence purpose",
                                description="Purpose:",
                                style={"description_width": "initial"},
                                layout=widgets.Layout(width="600px"))
limitations_input = widgets.Textarea(
    value="", placeholder="What does this index miss?",
    description="Limitations:",
    layout=widgets.Layout(width="600px", height="80px"),
    style={"description_width": "initial"})

# Helper to (re)build the per-variable controls when selections change.
def _rebuild_variable_controls(*_):
    selected = list(variable_picker.value)
    direction_toggles.clear()
    weight_sliders.clear()
    rationale_inputs.clear()
    direction_box.children = []
    weight_box.children    = []
    rationale_box.children = []
    if not selected:
        weight_total_label.value = "<b>Total weight: 0.00</b>"
        return

    direction_children = []
    weight_children    = []
    rationale_children = []
    default_weight = round(1.0 / len(selected), 2)
    for variable_name in selected:
        toggle = widgets.ToggleButtons(
            options=[("Higher = Better", "higher_better"),
                     ("Higher = Worse",  "higher_worse")],
            value="higher_better",
            description=variable_name,
            style={"description_width": "initial"})
        direction_toggles[variable_name] = toggle
        direction_children.append(toggle)

        slider = widgets.FloatSlider(
            value=default_weight, min=0.0, max=1.0, step=0.05,
            description=variable_name,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="600px"))
        weight_sliders[variable_name] = slider
        slider.observe(_on_weight_change, names="value")
        weight_children.append(slider)

        rationale = widgets.Text(
            value="", placeholder=f"Why is {variable_name} in this index?",
            description=variable_name,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="700px"))
        rationale_inputs[variable_name] = rationale
        rationale_children.append(rationale)

    direction_box.children = direction_children
    weight_box.children    = weight_children
    rationale_box.children = rationale_children
    _on_weight_change(None)

def _on_weight_change(_):
    total = sum(slider.value for slider in weight_sliders.values())
    if abs(total - 1.0) <= 0.01:
        color = "#117864"
    else:
        color = "#C0392B"
    weight_total_label.value = (
        f"<b style='color:{color}'>Total weight: {total:.2f}"
        f" {'(OK)' if abs(total-1.0) <= 0.01 else '— must equal 1.00'}</b>")

variable_picker.observe(_rebuild_variable_controls, names="value")
_rebuild_variable_controls()

# --- Step 6: run button + output area
run_button   = widgets.Button(description="Run my index", button_style="primary",
                              icon="play")
output_panel = widgets.Output()

def _format_methodology_card(payload):
    """Return a formatted plain-text card from the run payload."""
    lines = []
    lines.append("=" * 72)
    lines.append(f"METHODOLOGY CARD — {payload['index_name']}")
    lines.append("=" * 72)
    lines.append(f"Purpose      : {payload['purpose']}")
    lines.append(f"Geography    : {payload['geography'].upper()}")
    lines.append(f"Normalization: {payload['normalization']}")
    lines.append(f"  -> {payload['normalization_explanation']}")
    lines.append(f"Date generated: {payload['generated_at']}")
    lines.append("")
    lines.append("Variables, weights, directions:")
    for component in payload["components"]:
        direction_label = ("Higher = Better" if not component['invert']
                           else "Higher = Worse")
        lines.append(f"  - {component['variable']:<22} "
                     f"weight={component['weight']:.2f}  {direction_label}")
        if component.get("rationale"):
            lines.append(f"      rationale: {component['rationale']}")
    lines.append("")
    lines.append(f"Known limitations:")
    lines.append(f"  {payload['limitations'] or '(left blank by the author)'}")
    lines.append("")
    lines.append(f"Top {payload['top_n']} areas:")
    for area in payload["top_areas"]:
        lines.append(f"  {area['rank']:>2}. {area['name']:<45} {area['score']:.3f}")
    lines.append("")
    lines.append(f"Bottom {payload['top_n']} areas:")
    for area in payload["bottom_areas"]:
        lines.append(f"  {area['rank']:>2}. {area['name']:<45} {area['score']:.3f}")
    lines.append("=" * 72)
    return "\n".join(lines)

def _save_card_pdf(card_text, pdf_path):
    """Render the card text onto a single matplotlib page and save as PDF."""
    fig = plt.figure(figsize=(8.5, 11))
    fig.text(0.05, 0.97, card_text, family="monospace",
             fontsize=8, va="top", ha="left")
    plt.axis("off")
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

def _on_run_clicked(_):
    with output_panel:
        output_panel.clear_output()
        selected = list(variable_picker.value)
        if len(selected) < 3 or len(selected) > 8:
            print(f"Select between 3 and 8 variables (you selected {len(selected)}).")
            return
        total_weight = sum(s.value for s in weight_sliders.values())
        if abs(total_weight - 1.0) > 0.01:
            print(f"Weights must sum to 1.0. Current total: {total_weight:.2f}.")
            return

        chosen_geography = geography_picker.value
        gdf = nta_agg if chosen_geography == "nta" else tract_agg
        # Use NTA name where available, GEOID otherwise.
        name_column = "nta_name" if "nta_name" in gdf.columns else "GEOID"

        components = []
        for variable_name in selected:
            invert_value = (direction_toggles[variable_name].value == "higher_worse")
            components.append({
                "variable":  variable_name,
                "weight":    float(weight_sliders[variable_name].value),
                "invert":    invert_value,
                "rationale": rationale_inputs[variable_name].value.strip(),
            })

        norm_method = normalization_picker.value
        gdf = gdf.copy()
        gdf["my_index_score"] = compute_index(gdf, components, normalizer=norm_method)
        gdf_ranked = gdf.dropna(subset=["my_index_score"]).sort_values(
            "my_index_score", ascending=False)

        top_rows    = gdf_ranked.head(TOP_N)
        bottom_rows = gdf_ranked.tail(TOP_N).iloc[::-1]

        payload = {
            "index_name":               index_name_input.value.strip() or "Untitled Index",
            "purpose":                  purpose_input.value.strip() or "(no purpose given)",
            "geography":                chosen_geography,
            "normalization":            norm_method,
            "normalization_explanation": NORMALIZER_EXPLANATIONS[norm_method],
            "generated_at":             datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "components":               components,
            "limitations":              limitations_input.value.strip(),
            "top_n":                    TOP_N,
            "top_areas":    [{"rank": i+1, "name": str(r[name_column]),
                              "score": float(r["my_index_score"])}
                             for i, r in enumerate(top_rows.itertuples())],
            "bottom_areas": [{"rank": i+1, "name": str(r[name_column]),
                              "score": float(r["my_index_score"])}
                             for i, r in enumerate(bottom_rows.itertuples())],
        }

        card_text = _format_methodology_card(payload)
        print(card_text)

        slug = "".join(c if c.isalnum() else "_" for c in payload["index_name"])[:50] or "my_index"
        timestamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
        txt_path = os.path.join(OUTPUT_FOLDER, "cards", f"{slug}_{timestamp}.txt")
        pdf_path = os.path.join(OUTPUT_FOLDER, "cards", f"{slug}_{timestamp}.pdf")
        with open(txt_path, "w") as text_file:
            text_file.write(card_text)
        _save_card_pdf(card_text, pdf_path)
        print(f"\nSaved methodology card (text): {txt_path}")
        print(f"Saved methodology card (PDF) : {pdf_path}")

        # Choropleth + top/bottom bar chart.
        center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
        index_map = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")
        folium.Choropleth(
            geo_data=gdf.__geo_interface__,
            data=gdf, columns=[name_column, "my_index_score"],
            key_on=f"feature.properties.{name_column}",
            fill_color="YlGnBu", fill_opacity=0.8, line_opacity=0.2,
            legend_name=payload["index_name"],
        ).add_to(index_map)
        folium.GeoJson(
            gdf, style_function=lambda feature: {"fillOpacity": 0,
                                                 "color": "#333", "weight": 0.3},
            tooltip=folium.features.GeoJsonTooltip(
                fields=[name_column, "my_index_score"] +
                       [c["variable"] for c in components],
                aliases=["Area", "Score"] + [c["variable"] for c in components],
                localize=True),
        ).add_to(index_map)
        display(index_map)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
        axes[0].barh(top_rows[name_column], top_rows["my_index_score"], color="#3C4ED6")
        axes[0].invert_yaxis(); axes[0].set_title(f"Top {TOP_N}")
        axes[1].barh(bottom_rows[name_column], bottom_rows["my_index_score"], color="#C0392B")
        axes[1].invert_yaxis(); axes[1].set_title(f"Bottom {TOP_N}")
        plt.tight_layout(); plt.show()

        # Persist the GeoDataFrame so Module 7 can pick it up.
        gdf.to_file(os.path.join(OUTPUT_FOLDER, "exports", "my_index.geojson"),
                    driver="GeoJSON")
        gdf.drop(columns="geometry").to_csv(
            os.path.join(OUTPUT_FOLDER, "exports", "my_index.csv"), index=False)
        with open(os.path.join(OUTPUT_FOLDER, "exports", "my_index_payload.json"),
                  "w") as payload_file:
            json.dump(payload, payload_file, indent=2)

run_button.on_click(_on_run_clicked)

display(widgets.VBox([
    widgets.HTML("<h4>Step 1 — Variables (select 3 to 8)</h4>"),
    variable_picker,
    widgets.HTML("<h4>Step 2 — Direction</h4>"),
    direction_box,
    widgets.HTML("<h4>Step 3 — Normalization method</h4>"),
    normalization_picker, normalization_help,
    widgets.HTML("<h4>Step 4 — Weights (must sum to 1.0)</h4>"),
    weight_box, weight_total_label,
    widgets.HTML("<h4>Variable rationales (one sentence each)</h4>"),
    rationale_box,
    widgets.HTML("<h4>Step 5 — Geography</h4>"),
    geography_picker,
    widgets.HTML("<h4>Step 6 — Metadata + Run</h4>"),
    index_name_input, purpose_input, limitations_input,
    run_button, output_panel,
]))

**Who benefits from this index? Who is disadvantaged by it?** If a city agency used this index to allocate funding, which communities would receive resources, and which would not? Is that the outcome you intended?

**Every ranking produces a bottom as well as a top. Who scores lowest in your index, and what does that mean for them?** If your index identifies "areas that need more trees," the low-scoring NTAs become a target list. Target lists get acted on. Are the residents of those areas asking for more trees, or are you assigning that priority to them?

**What would this index look like if it were built by the residents of the lowest-scoring areas rather than by planners or designers?** That is not a rhetorical question. It is a methodological prompt. If your final-project index never asks it, the index is incomplete.

---

## Module 7 — Export and Visualization

Final exports and a layered map that brings every module's outputs together.

**Module 7 depends on:** at minimum Modules 0-3 and a Module 6 run (we read `exports/my_index.geojson`).

In [ ]:
# Module 7 — final layered map and exports.
import os
import geopandas as gpd
import pandas as pd
import folium

# Load the student's index from Module 6.
student_index_path = os.path.join(OUTPUT_FOLDER, "exports", "my_index.geojson")
if not os.path.exists(student_index_path):
    print("No student index found at exports/my_index.geojson.")
    print("Run Module 6 first.")
    raise SystemExit

student_index = gpd.read_file(student_index_path).to_crs(epsg=4326)
name_column = "nta_name" if "nta_name" in student_index.columns else "GEOID"

# Subway + wifi points as toggleable layers.
subway_df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "subway_entrances.csv")) \
    if os.path.exists(os.path.join(OUTPUT_FOLDER, "data", "subway_entrances.csv")) else pd.DataFrame()
wifi_df   = pd.read_csv(os.path.join(OUTPUT_FOLDER, "data", "wifi_hotspots.csv"))

emergent_path = os.path.join(OUTPUT_FOLDER, "data", "emergent_boundaries.geojson")
emergent_gdf  = gpd.read_file(emergent_path).to_crs(epsg=4326) \
    if os.path.exists(emergent_path) else None

# Center on the geographic mean of the student's areas.
center = [student_index.geometry.centroid.y.mean(),
          student_index.geometry.centroid.x.mean()]
final_map = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")

# Base layer: the student's index choropleth.
folium.Choropleth(
    geo_data=student_index.__geo_interface__,
    data=student_index,
    columns=[name_column, "my_index_score"],
    key_on=f"feature.properties.{name_column}",
    fill_color="YlGnBu", fill_opacity=0.75, line_opacity=0.15,
    legend_name="My Index",
).add_to(final_map)

# Tooltip layer on top of choropleth.
folium.GeoJson(
    student_index,
    style_function=lambda feature: {"fillOpacity": 0, "color": "#333", "weight": 0.3},
    tooltip=folium.features.GeoJsonTooltip(
        fields=[name_column, "my_index_score"], aliases=["Area", "Score"]),
    name="Boundaries",
).add_to(final_map)

# Highlight top-N areas with a distinct border color.
top_n_rows = student_index.dropna(subset=["my_index_score"]) \
    .sort_values("my_index_score", ascending=False).head(TOP_N)
folium.GeoJson(
    top_n_rows,
    style_function=lambda feature: {"fillOpacity": 0,
                                    "color": "#3C4ED6", "weight": 3},
    tooltip=folium.features.GeoJsonTooltip(
        fields=[name_column, "my_index_score"],
        aliases=["Top area", "Score"]),
    name=f"Top {TOP_N} areas",
).add_to(final_map)

# Subway points layer.
if not subway_df.empty:
    subway_group = folium.FeatureGroup(name="Subway entrances", show=False)
    for _, row in subway_df.dropna(subset=["latitude", "longitude"]).iterrows():
        folium.CircleMarker(
            location=[float(row["latitude"]), float(row["longitude"])],
            radius=3, color="#1B1B33", fill=True, fill_opacity=0.7,
        ).add_to(subway_group)
    subway_group.add_to(final_map)

# Wi-Fi points layer.
if not wifi_df.empty:
    wifi_group = folium.FeatureGroup(name="Wi-Fi hotspots", show=False)
    for _, row in wifi_df.dropna(subset=["latitude", "longitude"]).iterrows():
        folium.CircleMarker(
            location=[float(row["latitude"]), float(row["longitude"])],
            radius=2, color="#16A085", fill=True, fill_opacity=0.6,
        ).add_to(wifi_group)
    wifi_group.add_to(final_map)

# Emergent 311 boundaries from Module 4.
if emergent_gdf is not None and not emergent_gdf.empty:
    folium.GeoJson(
        emergent_gdf,
        style_function=lambda feature: {"fillOpacity": 0.0,
                                        "color": "#E67E22", "weight": 1.2,
                                        "dashArray": "4, 3"},
        name="Emergent 311 boundaries",
        show=False,
    ).add_to(final_map)

folium.LayerControl(collapsed=False).add_to(final_map)

final_map_path = os.path.join(OUTPUT_FOLDER, "maps", "final_map.html")
final_map.save(final_map_path)
print(f"Saved final layered map: {final_map_path}")

# Re-export csv/geojson explicitly so they live in exports/.
csv_path     = os.path.join(OUTPUT_FOLDER, "exports", "my_index.csv")
geojson_path = os.path.join(OUTPUT_FOLDER, "exports", "my_index.geojson")
student_index.drop(columns="geometry").to_csv(csv_path, index=False)
student_index.to_file(geojson_path, driver="GeoJSON")
print(f"Saved: {csv_path}")
print(f"Saved: {geojson_path}")

display(final_map)

---

## Closing reflection

You have built an index. Before you use it, sit with these questions:

- **What would the residents of the lowest-scoring areas say about this index if they were in the room?** If your map labels their neighborhood "low cohesion" or "high stress," what does that label do for them — or to them?

- **What data does not exist in any of these datasets — and whose experience is therefore invisible in your map?** Undocumented residents, people in informal housing, people who do not call 311, people whose ACS form went unanswered. Every absence in the data is a presence in someone's life that your map cannot see.

- **What would it mean to build this index with communities rather than about them?** Co-production is not a methodology label — it is a relationship. What would change about your variable selection, weighting, and geography if the people you were measuring were the people making the choices?

An index is an argument. Make sure yours is the argument you would defend in a room full of the people it is about.
